# Apache Iceberg com Apache Spark

Este notebook demonstra o uso do **Apache Iceberg** integrado ao **Apache Spark (PySpark)**.

## Cenário
Tabela de **clientes** de uma loja fictícia, com operações de INSERT, UPDATE e DELETE usando o formato Iceberg.

## DDL — Definição da Tabela

```sql
CREATE TABLE clientes (
    id INT,
    nome STRING,
    cidade STRING,
    valor_compra DOUBLE
) USING iceberg;

## Modelo da Tabela
Tabela `clientes` com os campos:
- `id`: identificador único
- `nome`: nome do cliente
- `cidade`: cidade do cliente
- `valor_compra`: valor total de compras

## Diagrama ER

```
┌─────────────────────────┐
│        CLIENTES         │
├─────────────────────────┤
│ PK  id           INT    │
│     nome         STRING │
│     cidade       STRING │
│     valor_compra DOUBLE │
└─────────────────────────┘
```

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Iceberg Demo") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/home/jovyan/work/data/iceberg_warehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession com Iceberg iniciada com sucesso!")

SparkSession com Iceberg iniciada com sucesso!


## INSERT — Criando a tabela e inserindo dados

In [2]:
from pyspark.sql import Row

# Criar a tabela Iceberg via SQL
spark.sql("""
    CREATE OR REPLACE TABLE local.db.clientes (
        id INT,
        nome STRING,
        cidade STRING,
        valor_compra DOUBLE
    ) USING iceberg
""")

# Inserir dados
dados = [
    Row(id=1, nome="Ana Silva", cidade="São Paulo", valor_compra=1500.0),
    Row(id=2, nome="Bruno Costa", cidade="Rio de Janeiro", valor_compra=2300.0),
    Row(id=3, nome="Carla Souza", cidade="Curitiba", valor_compra=800.0),
    Row(id=4, nome="Diego Lima", cidade="Belo Horizonte", valor_compra=3200.0),
    Row(id=5, nome="Eva Martins", cidade="Porto Alegre", valor_compra=950.0),
]

df = spark.createDataFrame(dados)
df.writeTo("local.db.clientes").append()

print("Dados inseridos com sucesso!")
spark.table("local.db.clientes").show()

Dados inseridos com sucesso!
+---+-----------+--------------+------------+
| id|       nome|        cidade|valor_compra|
+---+-----------+--------------+------------+
|  1|  Ana Silva|     São Paulo|      1500.0|
|  2|Bruno Costa|Rio de Janeiro|      2300.0|
|  3|Carla Souza|      Curitiba|       800.0|
|  4| Diego Lima|Belo Horizonte|      3200.0|
|  5|Eva Martins|  Porto Alegre|       950.0|
+---+-----------+--------------+------------+



## UPDATE — Atualizando registros

Atualizando a cidade e valor de compra da cliente Ana Silva.

In [3]:
spark.sql("""
    UPDATE local.db.clientes
    SET cidade = 'Campinas', valor_compra = 1800.0
    WHERE id = 1
""")

print("UPDATE realizado com sucesso!")
spark.table("local.db.clientes").show()

UPDATE realizado com sucesso!
+---+-----------+--------------+------------+
| id|       nome|        cidade|valor_compra|
+---+-----------+--------------+------------+
|  1|  Ana Silva|      Campinas|      1800.0|
|  2|Bruno Costa|Rio de Janeiro|      2300.0|
|  3|Carla Souza|      Curitiba|       800.0|
|  4| Diego Lima|Belo Horizonte|      3200.0|
|  5|Eva Martins|  Porto Alegre|       950.0|
+---+-----------+--------------+------------+



## DELETE — Removendo registros

Removendo a cliente Carla Souza (id = 3).

In [4]:
spark.sql("""
    DELETE FROM local.db.clientes
    WHERE id = 3
""")

print("DELETE realizado com sucesso!")
spark.table("local.db.clientes").show()

DELETE realizado com sucesso!
+---+-----------+--------------+------------+
| id|       nome|        cidade|valor_compra|
+---+-----------+--------------+------------+
|  1|  Ana Silva|      Campinas|      1800.0|
|  2|Bruno Costa|Rio de Janeiro|      2300.0|
|  4| Diego Lima|Belo Horizonte|      3200.0|
|  5|Eva Martins|  Porto Alegre|       950.0|
+---+-----------+--------------+------------+



## Histórico de alterações (Snapshots)

O Iceberg mantém snapshots de cada operação realizada na tabela.

In [5]:
spark.sql("SELECT snapshot_id, committed_at, operation FROM local.db.clientes.snapshots").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|2002976626500604422|2026-06-02 19:22:53.643|append   |
|6700743775827206413|2026-06-02 19:27:20.774|overwrite|
|4520394231385089917|2026-06-02 19:27:31.043|delete   |
+-------------------+-----------------------+---------+



## Conclusão

O Apache Iceberg adiciona ao Apache Spark:
- Transações ACID
- Versionamento por snapshots
- Operações de UPDATE e DELETE com SQL padrão
- Compatibilidade com múltiplos engines (Spark, Flink, Trino)